In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torch.nn as nn
import torch.nn.functional as F


import json
import pickle
import numpy as np
from collections import defaultdict
import pandas as pd
import json
import pandas as pd
from torchvision.io import read_image
from collections import Counter, defaultdict

In [2]:
#for good data sets, only thing to change
file_path_philipp = '/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/preprocessing/pkl_files_filtered/'

data_set_name = 'treutlein_all_genes'

#input
hyper_params_input = data_set_name + '_test_accuracy_best_hyper_params.json'
model_weights_input = data_set_name + '_model_weights.pth'
test_data_set_input = data_set_name + '_test_dataset.pth'
barcode_NN_cont_mapping_input = data_set_name + '_barcode_NN_cont_mapping.pkl'
data_set_input = file_path_philipp + data_set_name + '.pkl'
#output
importance_scores_output = data_set_name + '_lime.csv'



Overview of barcode_cont distribution

In [3]:
class CustomDataset(Dataset):
    # pca and occlusion manipulation outside of this class
    def __init__(self, dataframe, features = None, labels = None):
        self.dataframe = dataframe.copy()
        if features is not None and labels is not None:
            self.features = features
            self.labels = labels
        else:
            self.labels = self.dataframe['barcode_NN'].values
            self.features = self.dataframe.drop(['barcode_cont', 'barcode_NN'], axis=1).values

        
    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        """
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        sample = torch.tensor(self.features[idx], dtype=torch.float)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)  
        return sample, label
    
    def num_classes(self):
        """
        Returns the number of unique classes in the dataset.
        """
        return np.unique(self.labels).size
        
    def num_features(self):
        """
        Returns the number of features for each sample in the dataset.
        """
        return self.features.shape[1]  

In [4]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class CustomLayer(nn.Module):
    
    def __init__(self, size, dropout_rate, activation_func_name):
        super(CustomLayer, self).__init__()
        self.linear = nn.Linear(size, size)
        self.bn = nn.BatchNorm1d(size)
        self.dropout = nn.Dropout(dropout_rate)
        self.activation_func_name = activation_func_name  

        # Selecting the activation function
        if activation_func_name == 'relu':
            self.activation_func = nn.ReLU(inplace=False)
        elif activation_func_name == 'elu':
            self.activation_func = nn.ELU()
        elif activation_func_name == 'selu':
            self.activation_func = nn.SELU()
        elif activation_func_name == 'swish':
            self.activation_func = Swish()
        elif activation_func_name == 'mish':
            self.activation_func = nn.Mish()
        else:
            raise ValueError("Unsupported activation function")

        self.initialize_weights() 
            
    def initialize_weights(self):
        if self.activation_func_name in ['relu', 'swish', 'mish']:
            nn.init.kaiming_normal_(self.linear.weight, mode='fan_out', nonlinearity='relu')
        elif self.activation_func_name in ['elu', 'selu']:
            nn.init.kaiming_normal_(self.linear.weight, mode='fan_out', nonlinearity='leaky_relu')
        nn.init.constant_(self.linear.bias, 0)
         
    def forward(self, x):
        identity = x
        out = self.linear(x)

        out = self.bn(out)
        out = self.activation_func(out)
        out = self.dropout(out)
        out = out + identity  
        return out

class GeneExpressionModel(nn.Module):
         
    def __init__(self, input_dimension, num_classes, hidden_units, dropout_rate, num_layers, activation_function):
        super(GeneExpressionModel, self).__init__()

        # First layer to transform input dimension to hidden_units dimension
        self.initial_layer = nn.Linear(input_dimension, hidden_units)

        # Create a list of custom layers with residual connections
        self.layers = nn.ModuleList([CustomLayer(hidden_units, dropout_rate, activation_function) for _ in range(num_layers - 1)])
        
        # Output layer
        self.output = nn.Linear(hidden_units, num_classes) 

        with torch.no_grad():
            # Xavier initialization for the initial layer
            nn.init.kaiming_normal_(self.initial_layer.weight)
            nn.init.constant_(self.initial_layer.bias, 0)
           
            # Scaled Kaiming initialization for the output layer
            nn.init.kaiming_normal_(self.output.weight, mode='fan_out', nonlinearity='relu')
            self.output.weight.data *= 0.01  # Scaling factor
            nn.init.constant_(self.output.bias, 0)  

    def forward(self, x):
        # Pass the input through the initial layer
        x = self.initial_layer(x)

        # Pass the input through each custom layer
        for layer in self.layers:
            x = layer(x)

        # Output layer
        x = self.output(x)
        return x


Load the best neural network and the test data set that we will analyze.

In [5]:
with open(hyper_params_input, 'r') as f:
    best_hyperparameters = json.load(f)

# initialize model and data loaders
best_model = GeneExpressionModel(
    input_dimension=best_hyperparameters["data_overview"]["number_of_genes"],
    num_classes=best_hyperparameters["data_overview"]["number_of_barcodes"],
    hidden_units=best_hyperparameters["hyperparameters"]["hidden_units"],
    dropout_rate=best_hyperparameters["hyperparameters"]["dropout_rate"],
    num_layers=best_hyperparameters["hyperparameters"]["num_layers"],
    activation_function=best_hyperparameters["hyperparameters"]["activation_function"]
)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

test_dataset = torch.load(test_data_set_input)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
best_model.load_state_dict(torch.load(model_weights_input))
best_model.to(device)


/tmp/ipykernel_3976366/1871324318.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_dataset = torch.load(test_data_set_input)
/tmp/ipykernel_3976366/1871324318.py:21

GeneExpressionModel(
  (initial_layer): Linear(in_features=20856, out_features=148, bias=True)
  (layers): ModuleList()
  (output): Linear(in_features=148, out_features=23, bias=True)
)

In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import torch


# Load the data and prepare the gene names
gene_matrix = pd.read_pickle(data_set_input)
gene_names = list(gene_matrix.drop(['barcode_cont'], axis=1).columns)

# Define the kernel function
def rbf_kernel(d, kernel_width):
    return np.sqrt(np.exp(-(d ** 2) / kernel_width ** 2))

# Define a custom LIME implementation
class CustomLIME:
    def __init__(self, model, training_data, feature_names, num_classes, kernel_width=1, noise_scaling=1.0):
        self.model = model
        self.training_data = training_data
        self.feature_names = feature_names
        self.num_classes = num_classes
        self.kernel_width = kernel_width
        self.noise_scaling = noise_scaling
        self.scaler = StandardScaler().fit(training_data)

    def kernel(self, d):
        return np.sqrt(np.exp(-(d ** 2) / self.kernel_width ** 2))

    def perturb_instance(self, instance, num_samples=5000):
        feature_stds = np.std(self.training_data, axis=0)
        perturbations = np.random.normal(scale=feature_stds * self.noise_scaling * 0.01, size=(num_samples, instance.shape[0]))
        synthetic_data = instance + perturbations
        return synthetic_data

    def explain_instance(self, instance):
        instance_scaled = self.scaler.transform([instance])
        synthetic_data = self.perturb_instance(instance_scaled[0])
        distances = np.linalg.norm(synthetic_data - instance_scaled, axis=1)
        weights = self.kernel(distances)

        synthetic_data_original_scale = self.scaler.inverse_transform(synthetic_data)
        
        # Convert to tensor properly
        input_tensor = torch.tensor(synthetic_data_original_scale, dtype=torch.float32).to(device)

        # Get model predictions
        with torch.no_grad():
            predictions = self.model(input_tensor)

        feature_importances_dict = {}
        for class_idx in range(self.num_classes):
            class_weights = (predictions.argmax(axis=1) == class_idx).astype(float) * weights
            if np.sum(class_weights) == 0:
                continue

            linear_model = Ridge(alpha=1, fit_intercept=True)
            linear_model.fit(synthetic_data, predictions[:, class_idx], sample_weight=class_weights)

            feature_importances = np.abs(linear_model.coef_)
            feature_importances = feature_importances / feature_importances.sum()

            feature_importances_dict[class_idx] = {self.feature_names[i]: feature_importances[i] for i in range(len(self.feature_names))}
        
        return feature_importances_dict

def predict_proba(input_data):
    best_model.eval()
    input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = best_model(input_tensor)
        probabilities = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    return probabilities




Here code for lime starts

In [7]:


with open(barcode_NN_cont_mapping_input, 'rb') as f:
    barcode_mapping = pickle.load(f)
    
# Extract features and labels from the test dataset
features = test_dataset.features
labels = test_dataset.labels

# Convert features to a pandas DataFrame if not already
if not isinstance(features, pd.DataFrame):
    features = pd.DataFrame(features)

# Initialize CustomLIME explainer
explainer = CustomLIME(model=predict_proba, training_data=features.values, feature_names=gene_names, num_classes=best_hyperparameters["data_overview"]["number_of_barcodes"], noise_scaling=1.0)

# Initialize defaultdict for storing feature importances and cell counts
# Outer defaultdict for barcode classes, inner defaultdict initialized with 0.0 for each gene
aggregated_importances = defaultdict(lambda: defaultdict(float))
cell_counts = defaultdict(int)

# Process each instance in the test dataset
for idx in range(len(test_dataset)):
    instance = features.iloc[idx].values
    explanation = explainer.explain_instance(instance)
    nn_barcode_class = labels[idx]  
    barcode_class = barcode_mapping[nn_barcode_class]  # M
    
    for gene in gene_names:
        aggregated_importances[barcode_class][gene]  # Accessing will initialize with 0.0 if not present

    # Update importances and cell counts
    for class_idx_nn, class_importances in explanation.items():
        class_idx_cont = barcode_mapping[class_idx_nn] 
        if class_idx_cont == barcode_class:
            for feature, importance in class_importances.items():
                aggregated_importances[barcode_class][feature] += importance
            cell_counts[barcode_class] += 1

# Normalize importances by the number of cells
normalized_importances = defaultdict(lambda: defaultdict(float))
for class_idx, feature_importances in aggregated_importances.items():
    for feature, importance in feature_importances.items():
        if cell_counts[class_idx] > 0:
            normalized_importances[class_idx][feature] = importance / cell_counts[class_idx]



# Prepare data for CSV
aggregated_data = []
for class_idx, feature_importances in normalized_importances.items():
    for gene_idx, (feature, importance) in enumerate(feature_importances.items()):
        aggregated_data.append({
            'barcode_cont': class_idx,
            'gene': feature,
            'gene_idx': gene_idx,
            'lime_score': importance
        })

# Convert to DataFrame
df_aggregated = pd.DataFrame(aggregated_data)

# Save to CSV
df_aggregated.to_csv(importance_scores_output, index=False)

# Print for debugging
print("Aggregated DataFrame:")
print(df_aggregated.head())




/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
/tmp/ipykernel_3976366/4206787018.py:70: UserWarning: To copy const

Aggregated DataFrame:
   barcode_cont        gene  gene_idx  lime_score
0          1090  FO538757.2         0    0.000084
1          1090  AP006222.2         1    0.000055
2          1090   LINC00115         2    0.000041
3          1090      FAM41C         3    0.000015
4          1090      SAMD11         4    0.000047


In [8]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv(importance_scores_output)
print(df)

# Group by 'gene' and 'gene_idx' and aggregate 'lime_score'
aggregated_df = df.groupby(['gene', 'gene_idx'], as_index=False).agg({'lime_score': 'sum'})

# Add the 'barcode_cont' column with value -1
aggregated_df['barcode_cont'] = -1

# Reorder the columns to match the original CSV structure
aggregated_df = aggregated_df[['barcode_cont', 'gene', 'gene_idx', 'lime_score']]

# Append the aggregated results to the original DataFrame
result_df = pd.concat([df, aggregated_df], ignore_index=True)

# Save the updated DataFrame back to the CSV file
result_df.to_csv(importance_scores_output, index=False)


        barcode_cont        gene  gene_idx  lime_score
0               1090  FO538757.2         0    0.000084
1               1090  AP006222.2         1    0.000055
2               1090   LINC00115         2    0.000041
3               1090      FAM41C         3    0.000015
4               1090      SAMD11         4    0.000047
...              ...         ...       ...         ...
458827          1045  AC145212.2     20851    0.000000
458828          1045  AC007325.2     20852    0.000000
458829          1045  AL354822.1     20853    0.000020
458830          1045        eGFP     20854    0.008053
458831          1045      Tomato     20855    0.000883

[458832 rows x 4 columns]
